<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/01-foundations/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [9]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [10]:
# TODO
# 1. Add the revenue column
df['revenue'] = df['qty'] * df['price']

# 2. Compute summary
row_count = len(df)
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

# 3. Report results
print(f"Total Rows:    {row_count}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units:   {total_units:,}")

Total Rows:    400
Total Revenue: $8,520.00
Total Units:   783


I created a new column called revenue by multiplying the item quantity by its price for each row, then added up all the rows to find the total money made and total items sold across the sale records. It was found that 400 transactions generated $8520 in total sales across 783 units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [11]:
# TODO
# Group by category, calculate revenue, and add share of total percentage
by_category = df.groupby('category', as_index=False)['revenue'].sum()
by_category['share_of_total_%'] = (by_category['revenue'] / df['revenue'].sum()) * 100

# Sort from highest to lowest
by_category = by_category.sort_values(by='revenue', ascending=False).reset_index(drop=True)

print(by_category)

   category  revenue  share_of_total_%
0      Food   4293.0         50.387324
1     Merch   1771.5         20.792254
2     Drink   1554.0         18.239437
3  RainGear    901.5         10.580986


I grouped the dataset by item category and then summed up the revenue for each one, then it was sorted from highest to lowest. Then calculatred ech category's contribution as a percentage of overall sales, showing that food generated over half of the total revenue. Food led with $4293.00 which was around 50% of total revenue followed my merch, drink, and rain gear and their respective revenues and shares of the total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [12]:
# TODO
# Group by vendor to compute average order revenue and total order count
vendor_stats = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_order_revenue='mean', order_count='count')
    .sort_values(by='avg_order_revenue', ascending=False)
    .reset_index()
)

print(vendor_stats)

  vendor_id  avg_order_revenue  order_count
0      V-01          22.595745           94
1      V-18          21.750000          108
2      V-05          20.580645           93
3      V-10          20.314286          105


I grouped the data by vendor id and then calculated the average order revenue and the total order count for each. V-01 had the highest average order revenue with $22.60 across 94 orders, outperforming the other vendors.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [13]:
# TODO
# Calculate the percentage share of total revenue generated by Merch
merch_share = (df[df['category'] == 'Merch']['revenue'].sum() / df['revenue'].sum()) * 100

print(f"{merch_share:.1f}%")

20.8%


Merch generated 20.8% of total revenue, accounts for about one-fifth of all sales dollars. What I did here was isolated all transactions labeled as merch to sum their total sales and then divided that figure by the overall revenue. Then multiplied it by 100 and rounded the decimal to get the appropriate percentage.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [14]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Identify unmatched vendor
unmatched_vendor = joined[joined['vendor_name'].isna()]['vendor_id'].unique()[0]

# Fill missing vendor name
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')

# Verification checks
print(f"Unmatched Vendor ID: {unmatched_vendor}")
print(f"Row count intact: {len(joined) == 400} ({len(joined)} rows)")
print(f"Revenue intact: {joined['revenue'].sum() == 8520.0} (${joined['revenue'].sum():,.2f})")

Unmatched Vendor ID: V-18
Row count intact: True (400 rows)
Revenue intact: True ($8,520.00)


** **bold text**The unmatched vendor, and what I did about it:** _..._

V-18 was missing from the lookup table, so I performed a left join to preserve all 400 transaction rows without dropping sales revenue, and assigned a fallback label ('Unknown Vendor') to keep the reporting clean. As well, I performed the left merge with the many_to_one validation to map vendor names onto each order wtihout dropping or duplicating anything. Then I verified that the total row count was 400 and the overal revenue was 8520 and remained unchanged.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [15]:
# TODO
pivot_df = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot_df)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown Vendor    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


I created a pivot table using pivot_table() to break down total sales revenue across every combination of vendor and product category as well as setting fill_value=0 to handle missing pairs better. By enabling margins=True I added automatic row and columm totals to confirm that all sales correcrlt reconcile to 8520 across the entire data set.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [16]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A: I would tell these vendors to shift their inventory mix towards food as it was the highest in demand and made up over 50% of the total revenue sales. In contrast, stand managers should scale back on rain gear stock as it had underperformed at various locations and only accounted for about 10% of total revenue sales. Additionally, stadium manahement should update the registry to onboard V18. Despite it beinf listed as unknown in the lookup table, it was still a top grossing vendor on the concourse. Having accurate cataloging of this high performing vendor is essential for accurate inventory planning and operational tracking during the next event.
B: Question 3 is the least trustworthy answer in my opinion due to several data limitations. First, the dataset's single highest grossing entity is completely missing from the vendor lookup table, making it impossible to evaluate true organizational performance when the top seller remains an anonymous fallback. Second, broad category definitions like "Merch" combine drastically different products priced between 4.50 and 24.00, which masks whether high revenue is driven by strong sales volume or a few high priced transactions. As well with only about 100 total transactions per vendor, these small sample sizes mean average order values can easily be skewed by a handful of unusually large purchases rather than reflecting consistent customer demand.